# LLM-Zoomcamp Homework: Agentic RAG

Walks through Q1–Q3: load lessons, index them, run a RAG query, read token usage.

## Setup

In [1]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=os.getenv('BASE_URL'),
    http_client=httpx.Client(verify=False), #privateCA
)

## Q1. How many lesson pages?

In [2]:
from ingest import load_data, build_index

documents = load_data()
print(f'Number of lesson pages: {len(documents)}')

Number of lesson pages: 72


## Q2. Indexing and searching

Query: *How does the agentic loop keep calling the model until it stops?*

In [3]:
index = build_index(documents)

query = 'How does the agentic loop keep calling the model until it stops?'
results = index.search(query, num_results=5)

print('First result filename:', results[0]['filename'])

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG — count input tokens

In [4]:
from rag_helper import RAG

rag = RAG(index=index, openai_client=openai_client, model='devstral')
result = rag.rag(query)

print('Answer:')
print(result.answer)
print()
print('Usage:', result.usage)

usage = result.usage
input_tokens = getattr(usage, 'prompt_tokens', None) or getattr(usage, 'input_tokens', None)
print(f'Input tokens: {input_tokens}')

Answer:
The agentic loop keeps calling the model using a `while` loop (specifically a `while True` loop) that continues until the model returns a response without any function calls.

Here is the detailed process of how it works:
1. **Call the Model:** The loop sends the current message history to the model.
2. **Check for Function Calls:** The code iterates through the model's response. If it finds an item with the type `function_call`, a flag (such as `has_function_calls`) is set to `True`.
3. **Execute and Append:** The code executes the requested function call via a helper, and the result is appended to the message history.
4. **Repeat or Stop:** 
    * If `has_function_calls` is `True`, the loop continues to the next iteration, sending the updated history (including the tool output) back to the model.
    * If `has_function_calls` is `False` (meaning the model provided a final answer without requesting any more tools), the loop hits a `break` statement and stops.

Usage: Completio

## Q4. Chunking


In [6]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))


295
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [ ]:


## Q5. RAG with chunking


In [9]:

chunk_index = build_index(chunks)
chunk_rag = RAG(index=chunk_index, openai_client=openai_client, model='devstral')
result_chunked = chunk_rag.rag(query)
print('Result RAG chunked:', result_chunked.usage.prompt_tokens)
print('Result without RAG:', result.usage.prompt_tokens)

Result RAG chunked: 2615
Result without RAG: 7963


## Q6. Turning it into an agent

Give the LLM a `search` tool and let it decide when to use it. Just run python agent.py to see the results.
